[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch06/NN_DL_ch06_FinBERT/NN_DL_ch06_FinBERT.ipynb)

# NN_DL_ch06_FinBERT

**Financial Sentiment Analysis with FinBERT**

Use HuggingFace FinBERT model to classify sentiment of real financial headlines. Visualize sentiment distribution and attention patterns.

In [ ]:
!pip install -q transformers torch matplotlib scikit-learn

In [ ]:
# Style & Reproducibility
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MAIN_BLUE = '#1A3A6E'
IDA_RED   = '#CD0000'
FOREST    = '#2E7D32'
CRIMSON   = '#DC3545'
AMBER     = '#FFC107'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 14,
    'figure.figsize': (10, 6),
})
np.random.seed(42)

def save_fig(name):
    plt.savefig(f'{name}.pdf', bbox_inches='tight', dpi=300, transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', dpi=150, transparent=True)
    plt.show()
    print(f'Saved {name}.pdf and {name}.png')

In [ ]:
# Financial Headlines Dataset
headlines = [
    "Apple reports record quarterly revenue of $124 billion, beating estimates",
    "Federal Reserve signals potential interest rate cuts in coming months",
    "Tesla stock plunges 12% after disappointing delivery numbers",
    "JPMorgan Chase posts strong earnings amid rising interest income",
    "Inflation rises to 3.7%, above economist expectations",
    "Amazon announces massive layoffs affecting 18,000 workers",
    "Bitcoin surges past $60,000 as institutional demand grows",
    "Bank of England holds rates steady, signals cautious optimism",
    "Nvidia market cap exceeds $1 trillion on AI chip demand",
    "Deutsche Bank reports unexpected loss, shares tumble",
    "US GDP growth accelerates to 4.9% in third quarter",
    "Oil prices drop sharply on demand concerns from China",
    "Microsoft cloud revenue grows 29%, beating expectations",
    "Credit Suisse collapse sends shockwaves through banking sector",
    "Gold prices hit all-time high as investors seek safe haven",
    "Meta cuts 11,000 jobs in largest tech layoff of the year",
    "S&P 500 closes at record high driven by tech rally",
    "European Central Bank raises rates by 50 basis points",
    "Walmart beats earnings expectations with strong holiday sales",
    "Crypto exchange FTX files for bankruptcy amid fraud allegations",
    "Goldman Sachs profits drop 66% as dealmaking slows",
    "China GDP grows 5.2% in 2023, meeting government target",
    "Boeing stock falls after 737 MAX safety concerns resurface",
    "Berkshire Hathaway cash pile reaches record $157 billion",
    "Unemployment falls to 3.4%, lowest level in 54 years",
    "Citigroup announces restructuring, plans to cut 20,000 jobs",
    "Google parent Alphabet beats revenue estimates, shares jump",
    "US national debt surpasses $34 trillion for first time",
    "Netflix gains 13 million subscribers, stock surges 15%",
    "Real estate market shows signs of cooling as mortgage rates rise",
    "Ford reports $1.3 billion loss on electric vehicle division",
    "Eurozone narrowly avoids recession with 0.1% growth",
    "Visa reports record payment volumes during holiday season",
    "Silicon Valley Bank collapse triggers fears of banking crisis",
    "Indian stock market hits record high on strong economic data",
    "Disney streaming losses narrow as password sharing crackdown begins",
    "OPEC agrees to cut oil production by 2 million barrels per day",
    "Samsung profits plunge 69% amid global chip downturn",
    "US consumer confidence rises to highest level in two years",
    "Binance CEO pleads guilty to money laundering violations",
    "Toyota remains world largest automaker with record sales",
    "UK inflation falls below 4% for first time in two years",
    "PayPal stock drops 12% on weak forward guidance",
    "Swiss National Bank posts record loss of $143 billion",
    "Alibaba plans to split into six business units",
    "US housing starts fall to lowest level since pandemic",
    "AMD shares surge on new AI chip announcement rivaling Nvidia",
    "UBS completes emergency acquisition of Credit Suisse",
    "Brazilian real strengthens as central bank holds rates",
    "Pfizer revenue drops 42% as COVID vaccine demand wanes",
]

true_labels = [
    'positive', 'positive', 'negative', 'positive', 'negative',
    'negative', 'positive', 'neutral',  'positive', 'negative',
    'positive', 'negative', 'positive', 'negative', 'positive',
    'negative', 'positive', 'neutral',  'positive', 'negative',
    'negative', 'positive', 'negative', 'positive', 'positive',
    'negative', 'positive', 'negative', 'positive', 'negative',
    'negative', 'neutral',  'positive', 'negative', 'positive',
    'positive', 'negative', 'negative', 'positive', 'negative',
    'positive', 'positive', 'negative', 'negative', 'neutral',
    'negative', 'positive', 'neutral',  'positive', 'negative',
]

print(f"Headlines: {len(headlines)}")
print(f"Labels: {len(true_labels)}")

In [ ]:
# Run FinBERT Sentiment Analysis
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

model_name = 'ProsusAI/finbert'
tokenizer = AutoTokenizer.from_pretrained(model_name)
finbert_model = AutoModelForSequenceClassification.from_pretrained(model_name)
pipe = pipeline('sentiment-analysis', model=finbert_model, tokenizer=tokenizer,
                truncation=True, max_length=512)

results = pipe(headlines)
pred_labels = [r['label'] for r in results]
pred_scores = [r['score'] for r in results]

print("Sample predictions:")
for i in range(10):
    print(f"  [{pred_labels[i]:8s} {pred_scores[i]:.3f}] {headlines[i][:60]}...")

In [ ]:
# Sentiment Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

labels = ['positive', 'neutral', 'negative']
cm = confusion_matrix(true_labels, pred_labels, labels=labels)

fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(cm, display_labels=['Positive', 'Neutral', 'Negative'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title('FinBERT Sentiment Classification', fontweight='bold')
save_fig('nn_ch06_sentiment_confusion')

print(classification_report(true_labels, pred_labels, labels=labels))

In [ ]:
# Attention Heatmap Visualization
import torch

sample_text = "Federal Reserve signals potential interest rate cuts in coming months"
inputs = tokenizer(sample_text, return_tensors='pt', truncation=True, max_length=64)
finbert_model.eval()
with torch.no_grad():
    outputs = finbert_model(**inputs, output_attentions=True)

attention = outputs.attentions[-1].squeeze().mean(dim=0).numpy()
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'].squeeze())

n = min(len(tokens), 20)
attention = attention[:n, :n]
tokens = tokens[:n]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(attention, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(n)); ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=10)
ax.set_yticks(range(n)); ax.set_yticklabels(tokens, fontsize=10)
plt.colorbar(im, ax=ax, label='Attention Weight')
ax.set_title('FinBERT Attention Heatmap (Last Layer, Averaged)', fontweight='bold')
plt.tight_layout()
save_fig('nn_ch06_attention_heatmap')

## Summary

Ran **FinBERT** sentiment analysis on 50 real financial headlines. Evaluated classification accuracy and visualized the transformer attention patterns.